# 10장 실습 — 배차와 할당 문제

호출 여러 건이 동시에 들어왔을 때 어느 차를 누구에게 보낼지 정하는 문제입니다. 교재 10장에 대응합니다.

이 실습의 뼈대는 3장과 같습니다.
느리지만 틀릴 데 없는 방법으로 정답지를 만들고, 그것으로 빠른 방법을 검증합니다.
여기서는 모든 경우를 세어 보는 완전탐색이 정답지이고, 헝가리안 알고리즘이 검증 대상입니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 비용행렬 (교재 10.1)

행이 승객, 열이 차량입니다. 칸 하나가 그 차가 그 승객에게 가는 데 걸리는 분입니다.
지금은 두 점 사이 직선거리를 시속 25km 로 나눈 값을 씁니다. 교재 10.1절과 같은 세 명, 세 대입니다.

In [ ]:
import numpy as np

from smartmob.teaching.dispatch import cost_matrix

passengers = [(37.539, 127.215), (37.545, 127.190), (37.552, 127.205)]     # (위도, 경도)
vehicles = [(37.541, 127.212), (37.560, 127.198), (37.535, 127.230)]

costs = cost_matrix(passengers, vehicles)      # costs[i, j] = 차량 j 가 승객 i 에게 가는 분
np.round(costs, 2)

표로 옮기면 이렇습니다.

| 분 | 차량 0 | 차량 1 | 차량 2 |
|---|---|---|---|
| 승객 0 | 0.8 | 6.7 | 3.4 |
| 승객 1 | 4.8 | 4.4 | 8.9 |
| 승객 2 | 3.3 | 2.6 | 7.0 |

행마다 가장 작은 값을 찾으면 승객 0은 차량 0, 승객 1과 승객 2는 둘 다 차량 1입니다. 겹칩니다.

## 2. 탐욕 배차 (교재 10.2)

가장 단순한 규칙입니다. 호출 순서대로, 남은 차 중 가장 가까운 것을 줍니다.

In [ ]:
from smartmob.teaching.dispatch import greedy_match, optimal_match

g = greedy_match(costs)
for m in g.matches:                      # Match(passenger, vehicle, cost) 가 하나씩 들어 있습니다
    print(f"승객 {m.passenger} ← 차량 {m.vehicle}  {m.cost:.2f}분")
print(f"\n총 대기 {g.total_cost:.2f}분")

승객 0이 차량 0을, 승객 1이 차량 1을 가져갑니다.
승객 2는 가장 가까운 차량 1이 없어졌으므로 7.0분 걸리는 차량 2를 받습니다. 총 12.14분입니다.

## 3. 할당 문제 (교재 10.3)

전체 대기시간의 합이 가장 작은 짝을 찾습니다.
승객 3명, 차량 3대면 경우의 수가 3! = 6가지뿐이라 전부 세어 볼 수 있습니다.
같은 문제를 `scipy` 의 헝가리안 구현(`optimal_match`)으로도 풀어 둘을 비교합니다.

In [ ]:
from itertools import permutations

# permutations(range(3)) 은 (0,1,2), (0,2,1), ... 여섯 가지 순서를 냅니다.
# p[i] 가 승객 i 에게 줄 차량 번호입니다.
best = min(permutations(range(3)), key=lambda p: sum(costs[i, p[i]] for i in range(3)))
print(f"완전탐색   짝 {best}, 총 대기 {sum(costs[i, best[i]] for i in range(3)):.2f}분")

o = optimal_match(costs)
for m in o.matches:
    print(f"승객 {m.passenger} ← 차량 {m.vehicle}  {m.cost:.2f}분")
print(f"헝가리안   총 대기 {o.total_cost:.2f}분 (탐욕은 {g.total_cost:.2f}분)")

승객 0이 가장 가까운 차량 0을 양보하고 차량 2를 받습니다. 0.8분이 3.4분이 됩니다.
대신 승객 1이 차량 0(4.8분), 승객 2가 차량 1(2.6분)을 받아 총 대기가 12.14분에서 10.72분으로 줄었습니다.
한 사람이 2.5분 손해 보고 나머지 둘이 4분 가까이 이득을 본 것입니다.

## 4. 헝가리안이 정말 최적인지 확인합니다

`optimal_match`는 SciPy의 `linear_sum_assignment`를 호출합니다. 현재 SciPy 문서에서는 수정 Jonker–Volgenant 알고리즘을 사용한다고 설명합니다.

확인 방법은 위와 같습니다. 모든 배정을 세어 보고 합이 가장 작은 것을 고릅니다.
5×5 면 120가지뿐입니다. 무작위 행렬 200개에서 두 답이 같은지 봅니다.
3장에서 다익스트라를 NetworkX 와 30쌍 맞춰 본 것과 같은 일입니다.

In [ ]:
def brute_force(costs):
    """모든 배정을 세어 보고 합이 가장 작은 것을 돌려줍니다.

    승객 수와 차량 수가 같은 정사각 행렬만 다룹니다.
    n! 가지를 전부 보므로 8×8 을 넘기면 쓸 수 없습니다. 실전용이 아니라 정답지용입니다.
    """
    n = costs.shape[0]
    best_order, best_total = None, float("inf")
    for order in permutations(range(n)):
        total = sum(costs[i, order[i]] for i in range(n))
        if total < best_total:
            best_order, best_total = order, total
    return best_order, best_total


rng = np.random.default_rng(42)          # 씨앗을 고정하면 매번 같은 행렬이 나옵니다

banner("헝가리안 vs 완전탐색 (5×5, 200회)")
worst_gap = 0.0
mismatch = 0
for _ in range(200):
    m = rng.uniform(1, 30, size=(5, 5))          # 1~30분 사이 무작위 비용
    _, brute_total = brute_force(m)
    gap = abs(brute_total - optimal_match(m).total_cost)
    worst_gap = max(worst_gap, gap)
    mismatch += gap > 1e-9                       # 부동소수점 오차보다 크면 다른 답

expect("합이 다른 경우", mismatch, 0)
print(f"    최대 차이 {worst_gap:.2e}분")

200번 전부 같습니다. 이제 헝가리안을 믿고 쓸 수 있습니다.

같은 정답지로 탐욕 배차를 재 보면 얼마나 손해인지 나옵니다.

In [ ]:
banner("탐욕 vs 완전탐색 (5×5, 200회)")
losses = []
for _ in range(200):
    m = rng.uniform(1, 30, size=(5, 5))
    _, best_total = brute_force(m)
    losses.append(greedy_match(m).total_cost / best_total - 1)    # 0.2 면 최적보다 20% 더 기다림

print(f"탐욕이 더 쓴 시간  평균 {np.mean(losses):.1%}, 최악 {np.max(losses):.1%}")
print(f"탐욕이 최적과 같았던 비율  {np.mean(np.array(losses) < 1e-9):.0%}")

평균 26% 더 기다리고 최악에는 두 배가 넘습니다. 다섯 번에 한 번만 같은 답입니다.
교재 10.4절은 하남 좌표로 만든 8×10 행렬이라 평균 14%, 최대 79% 로 조금 작습니다.
어느 쪽이든 먼저 부른 사람에게 가장 가까운 차를 주는 것이 전체로는 최선이 아닙니다.

## 5. 완전탐색을 쓸 수 없는 이유

정답지는 작은 문제에서만 만들 수 있습니다. 경우의 수가 계승(n!)으로 늘어나기 때문입니다.

In [ ]:
import math

import pandas as pd

rows = [{"n": n, "경우의 수": math.factorial(n)} for n in [3, 5, 8, 10, 12, 15, 20]]
table = pd.DataFrame(rows)
table["년 (1초에 100만 가지)"] = (table["경우의 수"] / 1e6 / (365.25 * 86400)).round(2)
table

승객 20명이면 243경 가지입니다. 1초에 100만 가지를 봐도 7만 7천 년이 걸립니다.
헝가리안은 승객이 두 배가 되면 계산이 여덟 배쯤 늘어나는 정도라 수백 명까지 문제없습니다.

## 6. 얼마나 빠른가 (교재 10.5)

느릴 것 같지만 재 봅니다. 크기를 5부터 300까지 키워 가며 두 방법의 시간을 잽니다.

In [ ]:
import time

banner("크기별 실행시간")
for n in [5, 20, 100, 300]:
    m = rng.uniform(1, 30, size=(n, n))
    t0 = time.perf_counter(); optimal_match(m); t_hung = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); greedy_match(m);  t_greedy = (time.perf_counter() - t0) * 1000
    print(f"{n:4d}×{n:<4d}  헝가리안 {t_hung:7.2f} ms   탐욕 {t_greedy:7.2f} ms")

100×100 부터 탐욕이 오히려 느립니다. 우리 탐욕 배차는 파이썬 이중 루프이고, `scipy` 의 헝가리안은 C 로 짜여 있습니다.
더 좋은 답을 더 빨리 내므로 실제 시뮬레이터의 기본값도 헝가리안입니다.

## 7. 픽업 소요시간 계산 방법 (교재 10.6)

지금까지 비용은 직선거리 ÷ 25km/h 였습니다. 세 가지를 비교합니다.
직선거리, 9장의 ETA 모델, 3장의 실제 라우팅입니다.
먼저 9장과 같은 데이터로 모델을 학습하고 도로망을 읽습니다.

In [ ]:
import lightgbm as lgb

from smartmob.data import data_path, load_road_graph
from smartmob.teaching.eta import FEATURES, TARGET

eta = pd.read_parquet(data_path("hanam/eta_samples.parquet"))
model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, random_state=42, verbose=-1)
model.fit(eta[FEATURES], eta[TARGET])          # 여기서는 검증이 목적이 아니라 2만 건 전부로 학습합니다

G = load_road_graph("hanam", modes=("drive",))
print(f"모델 학습 완료, 도로망 노드 {G.n_nodes:,}개")

무작위 점은 시 경계 밖에 떨어져 도로로 닿지 않을 수 있습니다.
3.4절의 경로 없음이고, 라우팅 행렬에 무한대(`inf`)로 나타납니다.
그런 쌍이 있으면 다시 뽑아서 세 방법이 같은 승객·차량을 보게 합니다.

In [ ]:
import random

from smartmob.teaching.dispatch import cost_matrix_from_model, cost_matrix_from_router

rng = random.Random(3)                    # 교재 10.6절과 같은 표본이 나오는 씨앗

def random_points(n):
    """하남 일대를 감싸는 사각형 안의 무작위 점 n개."""
    return [(37.50 + rng.random() * 0.10, 127.13 + rng.random() * 0.14) for _ in range(n)]

while True:
    P, V = random_points(6), random_points(8)
    t0 = time.perf_counter()
    c_router = cost_matrix_from_router(P, V, G)
    t_r = time.perf_counter() - t0
    if np.isfinite(c_router).all():       # 도로로 못 닿는 쌍이 없을 때까지
        break

t0 = time.perf_counter(); c_straight = cost_matrix(P, V);                       t_s = time.perf_counter() - t0
t0 = time.perf_counter(); c_model = cost_matrix_from_model(P, V, model.predict); t_m = time.perf_counter() - t0

print(f"직선거리    {t_s * 1000:7.1f} ms")
print(f"ETA 모델    {t_m * 1000:7.1f} ms")
print(f"실제 라우팅 {t_r * 1000:7.1f} ms")

라우팅이 가장 느립니다. 6×8 = 48칸을 채우는 데만 이만큼 걸립니다.

세 방법이 같은 결정을 내리는지 봅니다. 중요한 것은 비용의 절댓값이 아니라 어느 차를 고르는가입니다.
그다음 각 결정을 실제 라우팅 비용으로 채점합니다.

In [ ]:
choice_straight = optimal_match(c_straight)
choice_model = optimal_match(c_model)
choice_router = optimal_match(c_router)

def pairs(result):
    """(승객, 차량) 짝의 집합. 집합끼리 & 로 겹치는 짝을 셉니다."""
    return {(m.passenger, m.vehicle) for m in result.matches}

def score(result):
    """그 배정을 실제 라우팅 비용으로 채점한 총 대기(분)."""
    return sum(c_router[m.passenger, m.vehicle] for m in result.matches)

n = len(pairs(choice_router))
banner("라우팅과 같은 짝을 고른 수 / 실제 총 대기")
print(f"직선거리  일치 {len(pairs(choice_straight) & pairs(choice_router))}/{n}   실제 총 대기 {score(choice_straight):.1f}분")
print(f"ETA 모델  일치 {len(pairs(choice_model) & pairs(choice_router))}/{n}   실제 총 대기 {score(choice_model):.1f}분")
print(f"라우팅    일치 {n}/{n}   실제 총 대기 {score(choice_router):.1f}분")

라우팅으로 정한 배차가 33.5분으로 가장 낫습니다. 채점 기준 자체가 라우팅 비용이므로 당연한 결과입니다.

중요한 것은 ETA 모델이 라우팅에 얼마나 가까운가입니다.
모델은 여섯 쌍 중 셋을 라우팅과 같게 골랐고, 실제 총 대기 34.9분으로 1.4분 차이입니다.
직선거리로 정한 배차는 50.9분으로 17분 넘게 손해입니다. 라우팅의 100분의 1 시간으로 이 정도면 배차에는 쓸 만합니다.

## 8. 빈칸

### 8.1 탐욕의 처리 순서

`greedy_match(costs, order=[...])` 로 처리 순서를 바꿀 수 있습니다.
5×5 행렬 하나를 놓고 순서를 무작위로 섞어 100번 돌려, 합계가 가장 좋았을 때와 나빴을 때를 구합니다.
먼저 부른 사람부터 처리하는 것이 좋은 규칙인지 한 줄로 적습니다.

In [ ]:
order_best = None       # 순서를 바꿔 얻은 가장 좋은 합계 (분)
order_worst = None      # 가장 나빴던 합계 (분)

banner("빈칸 8.1")
todo("가장 좋았던 합계", order_best, fmt=lambda v: f"{v:.2f}분")
todo("가장 나빴던 합계", order_worst, fmt=lambda v: f"{v:.2f}분")

### 8.2 승객과 차량 수가 다를 때

승객 5명에 차량 3대인 행렬을 만들어 `optimal_match` 를 돌립니다.
배차받지 못한 승객 번호가 결과의 `unmatched_passengers` 에 들어옵니다.
누가 남는지, 그것이 공평한지 두 줄로 적습니다.

In [ ]:
unmatched_count = None      # 배차받지 못한 승객 수

banner("빈칸 8.2")
todo("배차받지 못한 승객", unmatched_count)

### 8.3 직선거리와 라우팅이 다른 답을 내는 경우

1절의 세 명, 세 대로 `cost_matrix` 와 `cost_matrix_from_router` 행렬을 만듭니다.
각각에 `optimal_match` 를 돌려 배정을 비교합니다.
두 배정이 같으면 위치를 바꿔 가며 다른 배정이 나오는 경우를 하나 찾습니다.
하남은 한강과 검단산이 있어 그런 자리가 있습니다.

In [ ]:
found_case = None       # 배정이 갈린 (passengers, vehicles) 한 쌍

banner("빈칸 8.3")
todo("배정이 갈린 사례", found_case, fmt=lambda x: "찾았습니다")

## 정리

- 배차는 비용행렬 문제입니다. 행이 승객, 열이 차량, 칸이 도착 예상시간입니다
- 탐욕은 먼저 부른 사람부터 가장 가까운 차를 줍니다. 세 명 예제에서 12.14분, 할당 문제로 풀면 10.72분입니다
- 작은 문제에서 완전탐색으로 정답지를 만들면 남의 구현을 믿지 않고 검증할 수 있습니다
- 완전탐색은 n! 이라 정답지로만 쓰고, 실전은 헝가리안을 씁니다. `scipy` 구현이라 파이썬 탐욕보다 빠릅니다
- 비용을 무엇으로 재느냐가 결정을 바꿉니다. ETA 모델은 라우팅과 1.4분 차이, 직선거리는 17분 손해였습니다
- 11장 실습에서 이 배차를 1분마다 부르는 루프를 직접 짭니다